In [5]:
import numpy as np
import pandas as pd
import yfinance as yf

# Atsisiunciame duomenis ( pvz . NVDA ir BTC )
tickers = ["BTC-USD", "DX-Y.NYB", "GC=F", "BZ=F"]
df = yf.download(tickers, start="2014-01-01")

# Suvienodinti dazni, nes portfeli sudaro kripto ir derivatyvai
df = df.dropna()

# Apskaiciuojame grazas
returns = df["Close"].pct_change().dropna()


[*********************100%***********************]  4 of 4 completed


In [6]:
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)


In [7]:
# 1 užduotis
def compute_rsi(price_series: pd.Series, window: int = 14) -> pd.Series:
    delta = price_series.diff()

    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)

    avg_gain = gains.rolling(window=window, min_periods=window).mean()
    avg_loss = losses.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_golden_cross_features(
    asset_df: pd.DataFrame, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:

    asset_df["ma50"] = (
        asset_df["Close"].rolling(window=short_window, min_periods=short_window).mean()
    )
    asset_df["ma200"] = (
        asset_df["Close"].rolling(window=long_window, min_periods=long_window).mean()
    )

    state = (asset_df["ma50"] > asset_df["ma200"]).fillna(False)
    prev_state = state.shift(1, fill_value=False)

    asset_df["in_golden_cross"] = state
    asset_df["golden_cross_event"] = state & (~prev_state)
    asset_df["death_cross_event"] = (~state) & prev_state

    return asset_df


def build_cross_features(
    asset_series: pd.Series, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:
    asset_df = pd.DataFrame({"Close": asset_series}).copy()

    asset_df = compute_golden_cross_features(asset_df, short_window, long_window)

    asset_df["rsi14"] = compute_rsi(asset_df["Close"], window=14)

    return asset_df


def analyze_assets(df: pd.DataFrame, weights: list[float]) -> dict[str, pd.DataFrame]:
    close_prices = df["Close"]

    assert np.isclose(sum(weights), 1), "Svoriai turi susideti i 1"
    assert (
        len(weights) == close_prices.shape[1]
    ), "Svoriu skaicius turi buti toks pats kaip ir aktyvu"

    assets_data: dict[str, pd.DataFrame] = {}
    for asset in close_prices.columns:
        asset_series = close_prices[asset]
        assets_data[asset] = build_cross_features(asset_series)

    return assets_data


In [8]:
assets_data = analyze_assets(df, [0.5, 0.2, 0.1, 0.2])

btc = assets_data["BTC-USD"].copy()

btc[-20:]

,Close,ma50,ma200,in_golden_cross,golden_cross_event,death_cross_event,rsi14
Date,,,,,,,
2026-02-12,66221.843750,86439.233359,103405.821934,False,False,False,21.484507
2026-02-13,68857.843750,85989.386172,103278.687207,False,False,False,27.015589
2026-02-17,67494.218750,85468.714453,103145.121738,False,False,False,24.727628
2026-02-18,66425.320312,84954.388359,102994.786660,False,False,False,23.982195
2026-02-19,66957.523438,84505.783672,102845.023926,False,False,False,27.799029
2026-02-20,68005.421875,84053.088047,102711.310762,False,False,False,29.977119
2026-02-23,64616.738281,83491.588594,102550.382070,False,False,False,31.586330
2026-02-24,64080.042969,82932.770547,102385.620684,False,False,False,33.815128
2026-02-25,67960.125000,82441.746328,102209.214004,False,False,False,43.155061


In [9]:
# read data\BTC_news_articles.csv
btc_news_df = pd.read_csv("data/BTC_news_articles.csv", sep=';')
btc_news_df = btc_news_df[["title", "newsDatetime"]]
btc_news_df

,title,newsDatetime
0,"Bitcoin (BTC) Loses 200-Day MA, Tries to Hold ...",2021-08-18 09:50
1,Crypto Analyst Lark Davis on Bitcoin: ‘Still G...,2021-08-18 10:10
2,Where I am BUYING Bitcoin,2021-08-18 18:27
3,Twitter’s Jack Dorsey Is Now Mining Bitcoin. H...,2021-08-18 18:56
4,Bitcoin Trading Volume Sinks Without Decisive ...,2021-08-18 19:00
...,...,...
4868,"Despite Identical Monetary Tailwinds, Gold is ...",2025-12-03 10:00
4869,$93K And Climbing: Analysts Say Bitcoin’s Push...,2025-12-03 10:00
4870,Major US Bank Recommends Wealthy Clients Consi...,2025-12-03 10:15
4871,Here’s Bitcoin’s Next Big Target After $93K Br...,2025-12-03 10:17


In [10]:
# read data\gold_1.xlsx
gold_news_df_1 = pd.read_excel("data/gold_1.xlsx")
gold_news_df_2 = pd.read_excel("data/gold_2.xlsx")
gold_news_df_3 = pd.read_excel("data/gold_3.xlsx")


In [12]:
gold_news_df_1 = gold_news_df_1.rename(
    columns={'Title':'title', 'Submitted Date':'date'}
)

gold_news_df_2 = gold_news_df_2.rename(
    columns={'title':'title', 'date':'date'}
)

gold_news_df_3 = gold_news_df_3.rename(
    columns={'Title':'title', 'Published Date':'date'}
)

for df in (gold_news_df_1, gold_news_df_2, gold_news_df_3):
    df['date'] = pd.to_datetime(df['date'], utc=True)

In [13]:
all_gold = pd.concat([gold_news_df_1, gold_news_df_2, gold_news_df_3],
                     ignore_index=True)

all_gold = all_gold.sort_values('date')[["title", "date"]]

In [14]:
all_gold

,title,date
0,Comex Gold Speculators 'Miss the Move' as Bull...,2013-10-02 00:00:00+00:00
1,Gold Price Hits $3000 as US Consumer Sentiment...,2016-03-07 00:00:00+00:00
2,"Gold Hits New Dollar Record, $25 Off $3000, as...",2016-10-07 00:00:00+00:00
3,$33 Silver Cuts Gold/Silver Ratio as US Stocks...,2016-11-09 00:00:00+00:00
4,"Gold Rises, Silver Spikes as Stocks Slump Agai...",2016-11-10 00:00:00+00:00
...,...,...
427,Trump's Spending & Inflation 'Look Positive' f...,2025-03-11 00:00:00+00:00
428,Gold Trading Hits Shanghai Record on Trump Vic...,2025-03-12 00:00:00+00:00
429,Gold Prices Regain 'Key' 200-Day Moving Averag...,2025-03-13 00:00:00+00:00
430,"Gold Prices Up 20% in 2016 on Euro NIRP, ETF D...",2025-03-14 00:00:00+00:00


In [15]:
# read data\oil_sentiment_headlines.csv
oil_news_df = pd.read_csv("data/oil_sentiment_headlines.csv")[["date", "headline"]]

oil_news_df

,date,headline
0,2019-01-03,$83.7 billion in Iraq &apos; s oil revenues in...
1,2019-01-10,Study: soda destroys kidneys.
2,2019-01-14,American official: We want Qatar to challenge ...
3,2019-01-15,"Stone decor in houses... happiness, warmth and..."
4,2019-01-15,From inside Haftar prisons... shocking account...
...,...,...
11063,2026-03-10,Iran Vows To Fight On After Trump Hints At Ear...
11064,2026-03-10,"US Dollar Is Still the Dominant Currency, Tema..."
11065,2026-03-10,Trump Signals Possible End to Iran War; Oil sl...
11066,2026-03-10,Today will be 'most intense day' of strikes on...


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

# pip install --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio

# Load FinBERT model and tokenizer
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Force CPU usage for stability in this environment
finbert_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=-1,
 )

print("FinBERT model loaded successfully on CPU")

c:\Users\zabit\miniconda3\envs\finbert-cpu311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16792.92it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT model loaded successfully on CPU


In [ ]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

sample_text = "Oil prices stabilized after stronger demand outlook."
print(finbert_pipeline(sample_text))

torch version: 2.10.0+cpu
cuda available: False
[{'label': 'positive', 'score': 0.9492732286453247}]
